# Using arxiv_map library for processing arxiv + COMET data

Here, I use the arxiv_map library to process arxiv + COMET data. However, I am kind of building the library as I build this notebook (using this for interactive dev/stub testing).

In [5]:
%load_ext autoreload
%autoreload 2

from arxiv_map.indexing import ArxivMetadataIndex, CometDataset

In [6]:
!ls data/raw/

arxiv-metadata-oai-snapshot.json  kaggle_arxiv.zip
comet_data_orig.jsonl		  sample.json


In [7]:
comet_path = "data/raw/sample.json"
comet = CometDataset(dataset_path=comet_path)

arxiv_path = "data/raw/arxiv-metadata-oai-snapshot.json"
arxiv_index = ArxivMetadataIndex(metadata_path=arxiv_path)

In [8]:
_ = comet.load()

In [9]:
for row in comet.iter_rows():
    print(row)

{'arxiv_id': 'arXiv:2201.06484', 'doi': '10.48550/arxiv.2201.06484', 'version': 'v1', 'prediction': [{'name': 'Bibek S. Dhami', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Vasudevan Iyer', 'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA', 'ror_id': 'https://ror.org/01qz5mb56'}]}, {'name': 'Aniket Pant', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Ravi P. N. Tripathi', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Benjamin J. Lawrie', 'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA', 'ror_id': 'https://ror.org/01

In [10]:
comet_ids = [row["arxiv_id"] for row in comet.iter_rows()]
print(comet_ids)

for batch in comet.iter_batches(batch_size=2):
    print(batch)

['arXiv:2201.06484', 'arXiv:2511.15976']
[{'arxiv_id': 'arXiv:2201.06484', 'doi': '10.48550/arxiv.2201.06484', 'version': 'v1', 'prediction': [{'name': 'Bibek S. Dhami', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Vasudevan Iyer', 'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA', 'ror_id': 'https://ror.org/01qz5mb56'}]}, {'name': 'Aniket Pant', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Ravi P. N. Tripathi', 'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205', 'ror_id': 'https://ror.org/008s83205'}]}, {'name': 'Benjamin J. Lawrie', 'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 

In [11]:
comet_ids[0]

'arXiv:2201.06484'

In [ ]:
# getting matching rows
comet_rows = comet.find_by_ids([comet_ids[0]])
arxiv_rows = arxiv_index.find_by_ids([comet_ids]) # this is very slow... we definitely need to fix this...

In [14]:
comet_rows

{'2201.06484': {'arxiv_id': 'arXiv:2201.06484',
  'doi': '10.48550/arxiv.2201.06484',
  'version': 'v1',
  'prediction': [{'name': 'Bibek S. Dhami',
    'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205',
      'ror_id': 'https://ror.org/008s83205'}]},
   {'name': 'Vasudevan Iyer',
    'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA',
      'ror_id': 'https://ror.org/01qz5mb56'}]},
   {'name': 'Aniket Pant',
    'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205',
      'ror_id': 'https://ror.org/008s83205'}]},
   {'name': 'Ravi P. N. Tripathi',
    'affiliations': [{'affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205',
      'ror_id': 'https://ror.org/008s83205'}]},
   {'name': 'Benjamin J. Lawrie',
    'affiliations': [{'affiliation': 'Center for Nanophase Materials Sciences, 

Now we are ready to match/merge data to create enriched data

In [15]:
from arxiv_map.merge import CometArxivMerger

In [24]:
merge = CometArxivMerger().merge(
    comet_rows = [comet_rows],
    arxiv_metadata_by_id = arxiv_rows
)

Skipping 1 IDs found in COMET only (not in arXiv metadata): ['']


Skipping 1 IDs found in arXiv metadata only (not in COMET): ['2201.06484']


In [32]:
merger = CometArxivMerger()
results = merger.merge(list(comet.iter_rows()), arxiv_rows)

Skipping 1 IDs found in COMET only (not in arXiv metadata): ['2511.15976']


In [33]:
results

[{'arxiv_id': '2201.06484',
  'doi': '10.48550/arxiv.2201.06484',
  'version': 'v1',
  'authors': [{'raw_name': 'Bibek S. Dhami',
    'normalized_name': 'bibek s. dhami',
    'affiliations': [{'raw_affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205',
      'normalized_affiliation': 'Department of Physics, University of Alabama at Birmingham, AL 35205',
      'ror_id': 'https://ror.org/008s83205',
      'institution_key': 'ror:008s83205'}]},
   {'raw_name': 'Vasudevan Iyer',
    'normalized_name': 'vasudevan iyer',
    'affiliations': [{'raw_affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA',
      'normalized_affiliation': 'Center for Nanophase Materials Sciences, Oak Ridge National Laboratory, TN, 37831, USA',
      'ror_id': 'https://ror.org/01qz5mb56',
      'institution_key': 'ror:01qz5mb56'}]},
   {'raw_name': 'Aniket Pant',
    'normalized_name': 'aniket pant',
    'affiliations': [{'raw_affiliat

In [34]:
from arxiv_map.sql import ArxivMapSQLClient

client = ArxivMapSQLClient()  # uses DATABASE_URL

In [36]:
summary = client.upload_merged_rows(results, refresh_network=True)
# this function uses merged rows and uploads to all necessary tables. also, it calls a function to coerce the rows to DB expected format

In [ ]:
summary # now, the Aniket PL paper has been added to all necessary tables

UploadSummary(papers=1, authors=6, institutions=2, links=7)

In [ ]:
# NOTE: this does not include geocoding. we need to work on geocoding script now to add here.